# 02 — Running Local LLMs (Instructor Notebook)

This instructor notebook is an expanded, step-by-step version of the quick demo. Each section explains what we do, why it matters, and how to avoid common problems. The examples are intentionally CPU-friendly for workshops, with notes for GPU and local runtimes.

## Goals for this notebook

1. Run a minimal text-generation pipeline (CPU).
2. Understand tokenizer → model → generate flow.
3. Learn practical instructor talking points about CPU vs GPU and local backends (llama.cpp, Ollama).

### Step 1 — Verify environment (what/why)

What: Check that `transformers` and `torch` are available.
Why: If these packages are missing, model loading will fail — catching this early saves time.

In [2]:
# Environment quick check
import importlib, sys
packages = ['transformers', 'torch']
info = {}
for p in packages:
    spec = importlib.util.find_spec(p)
    info[p] = bool(spec)
print('Environment check:')
for k,v in info.items():
    print(f'- {k}:', 'installed' if v else 'MISSING')
print('Python:', sys.version.split()[0])

Environment check:
- transformers: installed
- torch: installed
Python: 3.12.7


### Step 2 — Run a small generation example (what/why)

What: Load `distilgpt2` and generate short text.
Why: This end-to-end exercise shows tokenization, model forward pass, and decoding — the essential inference flow.

In [3]:
# Minimal text generation (may download model on first run)
try:
    from transformers import pipeline
    gen = pipeline('text-generation', model='distilgpt2')
    prompt = 'In one sentence, explain DNA sequencing:'
    results = gen(prompt, max_length=80, num_return_sequences=1)
    print('Generated:')
    print(results[0]['generated_text'])
except Exception as e:
    print('Unable to run generation. Possible causes: no network to download model, or packages missing.')
    print('Error:', e)

/home/silvanopiazza/anaconda3_v2/lib/python3.12/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/silvanopiazza/anaconda3_v2/lib/python3.12/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/silvanopiazza/anaconda3_v2/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated:
In one sentence, explain DNA sequencing:

The results are so striking, with almost no real evidence. So far, the results are not good (and no one at all expected), but rather "interesting" (that the "DNA sequencing" approach works best at determining the exact sequence of DNA).
This is the case, since DNA sequencing has a lot to offer: the results are always


### What this cell demonstrates (explain)

- Tokenizer converts text to token IDs.
- Model produces logits for next tokens.
- Decoder converts token IDs back to text.

Instructor tip: show how to change `max_length` and explain trade-offs (longer = costlier + potential divergence).

### Step 3 — GPU and advanced options (what/why)

If a GPU is available, `torch.cuda.is_available()` will be True. Using a GPU speeds up inference and enables larger models. Alternatives: quantized GGUF models with `llama.cpp` or Ollama for managed local runtimes.

In [4]:
# Check CUDA (if installed)
try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('Device count:', torch.cuda.device_count())
        print('Device name:', torch.cuda.get_device_name(0))
except Exception as e:
    print('PyTorch not available or cannot check CUDA:', e)

CUDA available: True
Device count: 1
Device name: NVIDIA GeForce RTX 3050 Ti Laptop GPU


## Instructor talking points — local backends
- `llama.cpp` — fast, low-memory CPU inference using quantized GGUF models; good for offline workshops.
- `Ollama` — easy model management and API; cross-platform.
- `Hugging Face Transformers` — most flexible for research and integration into Python code.

## Exercise A (instructor-led) — Prompt engineering demo
Change the `prompt` in the generation cell to a different style: (1) very short question, (2) explicit instruction, (3) step-by-step prompt. Re-run and discuss differences in output style, length and correctness.
Suggested solution: Try 'Explain CRISPR in one sentence.' vs 'Explain CRISPR step by step for a beginner.'